1. 输入原始数据 DataFrame
2. 为每个 model / x / y 拟合响应曲线
3. 对 fitted curve 做 form classification
4. 在 monotonic increase/decrease 之后进一步判断 saturation
5. 输出完整结果表
6. 画图检查分类结果

In [2]:
# =========================================================
# Form classification with saturation
# =========================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from statsmodels.nonparametric.smoothers_lowess import lowess
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline


# =========================================================
# 1. Basic fitted-curve methods
# =========================================================

def fit_binned_curve(
    df,
    x_col,
    y_col,
    n_bins=30,
    bin_type="equal_count"
):
    """
    Fit a response curve using binned median values.

    Parameters
    ----------
    df : pandas.DataFrame
        Raw data containing x and y columns.
    x_col : str
        Predictor variable.
    y_col : str
        Response variable.
    n_bins : int
        Number of bins.
    bin_type : {"equal_count", "equal_width"}
        Binning strategy.

    Returns
    -------
    x_curve, y_curve : np.ndarray
        Fitted curve coordinates.
    """

    data = df[[x_col, y_col]].replace([np.inf, -np.inf], np.nan).dropna().copy()

    if len(data) < n_bins:
        return np.array([]), np.array([])

    data = data.sort_values(x_col)

    if bin_type == "equal_count":
        data["bin"] = pd.qcut(
            data[x_col],
            q=n_bins,
            duplicates="drop"
        )

    elif bin_type == "equal_width":
        data["bin"] = pd.cut(
            data[x_col],
            bins=n_bins,
            duplicates="drop"
        )

    else:
        raise ValueError("bin_type must be 'equal_count' or 'equal_width'.")

    curve = (
        data
        .groupby("bin", observed=True)
        .agg(
            x_median=(x_col, "median"),
            y_median=(y_col, "median"),
            n=(y_col, "size")
        )
        .reset_index(drop=True)
    )

    curve = curve[curve["n"] > 0]

    return curve["x_median"].values, curve["y_median"].values


def fit_lowess_curve(
    df,
    x_col,
    y_col,
    frac=0.25,
    n_grid=120,
    sample_size=20000,
    random_state=42
):
    """
    Fit a LOWESS response curve.

    LOWESS is useful for describing a smooth empirical relationship
    without assuming a parametric functional form.
    """

    data = df[[x_col, y_col]].replace([np.inf, -np.inf], np.nan).dropna().copy()

    if len(data) < 20:
        return np.array([]), np.array([])

    if len(data) > sample_size:
        data = data.sample(sample_size, random_state=random_state)

    data = data.sort_values(x_col)

    x = data[x_col].values
    y = data[y_col].values

    fitted = lowess(
        endog=y,
        exog=x,
        frac=frac,
        return_sorted=True
    )

    x_fit = fitted[:, 0]
    y_fit = fitted[:, 1]

    # remove duplicated x values
    unique_mask = np.concatenate([[True], np.diff(x_fit) > 0])
    x_fit = x_fit[unique_mask]
    y_fit = y_fit[unique_mask]

    if len(x_fit) < 5:
        return np.array([]), np.array([])

    # interpolate to a regular grid for stable slope calculation
    x_grid = np.linspace(np.nanmin(x_fit), np.nanmax(x_fit), n_grid)
    y_grid = np.interp(x_grid, x_fit, y_fit)

    return x_grid, y_grid


def fit_polynomial_curve(
    df,
    x_col,
    y_col,
    degree=2,
    n_grid=120,
    sample_size=30000,
    random_state=42
):
    """
    Fit a polynomial response curve.

    This is mainly used as a simple parametric comparison method.
    """

    data = df[[x_col, y_col]].replace([np.inf, -np.inf], np.nan).dropna().copy()

    if len(data) < 20:
        return np.array([]), np.array([])

    if len(data) > sample_size:
        data = data.sample(sample_size, random_state=random_state)

    x = data[x_col].values.reshape(-1, 1)
    y = data[y_col].values

    model = make_pipeline(
        PolynomialFeatures(degree=degree, include_bias=False),
        LinearRegression()
    )

    model.fit(x, y)

    x_grid = np.linspace(np.nanmin(x), np.nanmax(x), n_grid).reshape(-1, 1)
    y_grid = model.predict(x_grid)

    return x_grid.ravel(), y_grid


# =========================================================
# 2. Curve shape classifier
# =========================================================

def classify_curve_shape(
    x_curve,
    y_curve,
    y_raw_std=None,
    flat_threshold=0.05,
    slope_tol=0.03,
    monotonic_fraction=0.70,
    min_net_change_ratio=0.10,
    turning_amplitude_ratio=0.10,
    side_fraction_threshold=0.60,
    roughness_max=3.0,
    saturation_strength_ratio=0.50,
    saturation_min_early_slope=0.15,
):
    """
    Classify the shape of a fitted response curve.

    Shape labels
    ------------
    - Flat / negligible response
    - Monotonic increase
    - Monotonic saturating increase
    - Monotonic decrease
    - Monotonic saturating decrease
    - U-shaped
    - Inverted-U-shaped
    - Complex / uncertain nonlinear
    - Insufficient data

    Important
    ---------
    Saturation is assessed only after the curve has first been identified
    as mostly monotonic.
    """

    x = np.asarray(x_curve, dtype=float)
    y = np.asarray(y_curve, dtype=float)

    valid = np.isfinite(x) & np.isfinite(y)
    x = x[valid]
    y = y[valid]

    if len(x) < 5:
        return {"shape_type": "Insufficient data"}

    # Sort by x
    order = np.argsort(x)
    x = x[order]
    y = y[order]

    # Remove duplicated x values
    unique_mask = np.concatenate([[True], np.diff(x) > 0])
    x = x[unique_mask]
    y = y[unique_mask]

    if len(x) < 5:
        return {"shape_type": "Insufficient data"}

    x_range = float(np.nanmax(x) - np.nanmin(x))
    y_range_raw = float(np.nanmax(y) - np.nanmin(y))

    if x_range == 0:
        return {"shape_type": "Insufficient data"}

    # -----------------------------------------------------
    # S1. Flat / negligible response gate
    # -----------------------------------------------------
    raw_scale_available = (
        y_raw_std is not None
        and np.isfinite(y_raw_std)
        and y_raw_std > 0
    )

    if raw_scale_available:
        y_range_to_raw_std = y_range_raw / y_raw_std

        if y_range_to_raw_std < flat_threshold:
            return {
                "shape_type": "Flat / negligible response",
                "y_range_raw": y_range_raw,
                "y_raw_std": y_raw_std,
                "y_range_to_raw_std": y_range_to_raw_std,
                "raw_scale_available": True,
            }

    else:
        y_range_to_raw_std = np.nan

        if y_range_raw == 0:
            return {
                "shape_type": "Flat / negligible response",
                "y_range_raw": y_range_raw,
                "y_raw_std": y_raw_std,
                "y_range_to_raw_std": np.nan,
                "raw_scale_available": False,
            }

    # -----------------------------------------------------
    # S2. Normalize fitted curve
    # -----------------------------------------------------
    y_min = float(np.nanmin(y))
    y_range = float(np.nanmax(y) - y_min)

    if y_range == 0:
        return {
            "shape_type": "Flat / negligible response",
            "y_range_raw": y_range_raw,
            "y_raw_std": y_raw_std,
            "y_range_to_raw_std": y_range_to_raw_std,
            "raw_scale_available": raw_scale_available,
        }

    x_norm = (x - np.nanmin(x)) / x_range
    y_norm = (y - y_min) / y_range

    # -----------------------------------------------------
    # S3. Local slopes and directional signs
    # -----------------------------------------------------
    dx = np.diff(x_norm)
    dy = np.diff(y_norm)

    valid_dx = dx != 0
    dx = dx[valid_dx]
    dy = dy[valid_dx]

    if len(dx) < 4:
        return {"shape_type": "Insufficient data"}

    slopes = dy / dx

    signs = np.zeros_like(slopes, dtype=int)
    signs[slopes > slope_tol] = 1
    signs[slopes < -slope_tol] = -1
    signs[np.abs(slopes) <= slope_tol] = 0

    n_segments = len(signs)

    pos_fraction = float(np.sum(signs == 1) / n_segments)
    neg_fraction = float(np.sum(signs == -1) / n_segments)
    flat_fraction = float(np.sum(signs == 0) / n_segments)

    nonzero_signs = signs[signs != 0]

    if len(nonzero_signs) == 0:
        n_sign_changes = 0
        first_sign = 0
        last_sign = 0
    else:
        n_sign_changes = int(np.sum(nonzero_signs[1:] != nonzero_signs[:-1]))
        first_sign = int(nonzero_signs[0])
        last_sign = int(nonzero_signs[-1])

    # -----------------------------------------------------
    # S4. Overall direction
    # -----------------------------------------------------
    net_change = float(y_norm[-1] - y_norm[0])
    net_change_ratio = abs(net_change)

    total_variation = float(np.sum(np.abs(np.diff(y_norm))))
    roughness_ratio = total_variation / (abs(net_change) + 1e-9)

    mostly_increasing = (
        net_change > min_net_change_ratio
        and pos_fraction + flat_fraction >= monotonic_fraction
    )

    mostly_decreasing = (
        net_change < -min_net_change_ratio
        and neg_fraction + flat_fraction >= monotonic_fraction
    )

    # -----------------------------------------------------
    # S5. Saturation as a subtype of monotonicity
    # -----------------------------------------------------
    mid = len(slopes) // 2

    early_slopes = slopes[:mid]
    late_slopes = slopes[mid:]

    # For increasing curves, compare positive marginal responses
    early_pos_slopes = early_slopes[early_slopes > slope_tol]
    late_pos_slopes = late_slopes[late_slopes > slope_tol]

    # For decreasing curves, compare absolute negative marginal responses
    early_neg_slopes = early_slopes[early_slopes < -slope_tol]
    late_neg_slopes = late_slopes[late_slopes < -slope_tol]

    early_increase_strength = (
        float(np.mean(early_pos_slopes))
        if len(early_pos_slopes) > 0
        else 0.0
    )

    late_increase_strength = (
        float(np.mean(late_pos_slopes))
        if len(late_pos_slopes) > 0
        else 0.0
    )

    early_decrease_strength = (
        float(np.mean(np.abs(early_neg_slopes)))
        if len(early_neg_slopes) > 0
        else 0.0
    )

    late_decrease_strength = (
        float(np.mean(np.abs(late_neg_slopes)))
        if len(late_neg_slopes) > 0
        else 0.0
    )

    saturating_increase = (
        mostly_increasing
        and early_increase_strength >= saturation_min_early_slope
        and late_increase_strength <= saturation_strength_ratio * early_increase_strength
    )

    saturating_decrease = (
        mostly_decreasing
        and early_decrease_strength >= saturation_min_early_slope
        and late_decrease_strength <= saturation_strength_ratio * early_decrease_strength
    )

    # -----------------------------------------------------
    # S6. Major turning point
    # -----------------------------------------------------
    min_idx = int(np.argmin(y_norm))
    max_idx = int(np.argmax(y_norm))

    valley_is_internal = 0 < min_idx < len(y_norm) - 1
    peak_is_internal = 0 < max_idx < len(y_norm) - 1

    left_drop_to_min = float(y_norm[0] - y_norm[min_idx])
    right_rise_from_min = float(y_norm[-1] - y_norm[min_idx])

    left_rise_to_max = float(y_norm[max_idx] - y_norm[0])
    right_drop_from_max = float(y_norm[max_idx] - y_norm[-1])

    # U-shaped: left side mostly decreasing/flat, right side mostly increasing/flat
    left_signs_to_min = signs[:max(min_idx, 1)]
    right_signs_from_min = signs[min_idx:]

    left_neg_or_flat = (
        float(np.mean((left_signs_to_min == -1) | (left_signs_to_min == 0)))
        if len(left_signs_to_min) > 0
        else 0.0
    )

    right_pos_or_flat = (
        float(np.mean((right_signs_from_min == 1) | (right_signs_from_min == 0)))
        if len(right_signs_from_min) > 0
        else 0.0
    )

    u_shape_candidate = (
        valley_is_internal
        and left_drop_to_min >= turning_amplitude_ratio
        and right_rise_from_min >= turning_amplitude_ratio
        and left_neg_or_flat >= side_fraction_threshold
        and right_pos_or_flat >= side_fraction_threshold
    )

    # Inverted-U-shaped: left side mostly increasing/flat, right side mostly decreasing/flat
    left_signs_to_max = signs[:max(max_idx, 1)]
    right_signs_from_max = signs[max_idx:]

    left_pos_or_flat = (
        float(np.mean((left_signs_to_max == 1) | (left_signs_to_max == 0)))
        if len(left_signs_to_max) > 0
        else 0.0
    )

    right_neg_or_flat = (
        float(np.mean((right_signs_from_max == -1) | (right_signs_from_max == 0)))
        if len(right_signs_from_max) > 0
        else 0.0
    )

    inverted_u_candidate = (
        peak_is_internal
        and left_rise_to_max >= turning_amplitude_ratio
        and right_drop_from_max >= turning_amplitude_ratio
        and left_pos_or_flat >= side_fraction_threshold
        and right_neg_or_flat >= side_fraction_threshold
    )

    # -----------------------------------------------------
    # S7. Decision tree
    # -----------------------------------------------------
    if u_shape_candidate:
        shape_type = "U-shaped"

    elif inverted_u_candidate:
        shape_type = "Inverted-U-shaped"

    elif mostly_increasing:
        if saturating_increase:
            shape_type = "Monotonic saturating increase"
        else:
            shape_type = "Monotonic increase"

    elif mostly_decreasing:
        if saturating_decrease:
            shape_type = "Monotonic saturating decrease"
        else:
            shape_type = "Monotonic decrease"

    elif n_sign_changes >= 2 or roughness_ratio > roughness_max:
        shape_type = "Complex / uncertain nonlinear"

    else:
        shape_type = "Complex / uncertain nonlinear"

    return {
        "shape_type": shape_type,

        # S1
        "y_range_raw": y_range_raw,
        "y_raw_std": y_raw_std,
        "y_range_to_raw_std": y_range_to_raw_std,
        "raw_scale_available": raw_scale_available,

        # S3-S4
        "net_change": net_change,
        "net_change_ratio": net_change_ratio,
        "pos_fraction": pos_fraction,
        "neg_fraction": neg_fraction,
        "flat_fraction": flat_fraction,
        "n_sign_changes": n_sign_changes,
        "first_sign": first_sign,
        "last_sign": last_sign,
        "roughness_ratio": roughness_ratio,

        # S5 saturation
        "early_increase_strength": early_increase_strength,
        "late_increase_strength": late_increase_strength,
        "early_decrease_strength": early_decrease_strength,
        "late_decrease_strength": late_decrease_strength,
        "saturating_increase": saturating_increase,
        "saturating_decrease": saturating_decrease,
        "saturation_strength_ratio": saturation_strength_ratio,
        "saturation_min_early_slope": saturation_min_early_slope,

        # S6 turning point
        "valley_is_internal": valley_is_internal,
        "peak_is_internal": peak_is_internal,
        "left_drop_to_min": left_drop_to_min,
        "right_rise_from_min": right_rise_from_min,
        "left_rise_to_max": left_rise_to_max,
        "right_drop_from_max": right_drop_from_max,
        "left_neg_or_flat": left_neg_or_flat,
        "right_pos_or_flat": right_pos_or_flat,
        "left_pos_or_flat": left_pos_or_flat,
        "right_neg_or_flat": right_neg_or_flat,
        "u_shape_candidate": u_shape_candidate,
        "inverted_u_candidate": inverted_u_candidate,
    }


# =========================================================
# 3. Run form classification for one dataset
# =========================================================

def run_form_classification_for_dataset(
    df,
    model_name,
    x_vars=("precipitation", "lai"),
    y_vars=("evapotrans", "tran", "evspsblveg", "evspsblsoi"),
    methods=("equal_count_bin", "equal_width_bin", "lowess", "polynomial_2", "polynomial_3"),
    n_bins=30,
    lowess_frac=0.25,
    n_grid=120,
):
    """
    Run fitted-curve construction and form classification for one model dataset.

    Parameters
    ----------
    df : pandas.DataFrame
        Input data for one model.
    model_name : str
        Model name, e.g., "CLASSIC" or "LPJ-GUESS".
    x_vars : tuple/list
        Predictor variables.
    y_vars : tuple/list
        Response variables.
    methods : tuple/list
        Curve-fitting methods.

    Returns
    -------
    result_df : pandas.DataFrame
        Classification results.
    curve_dict : dict
        Stored fitted curves for plotting and checking.
    """

    rows = []
    curve_dict = {}

    for x_col in x_vars:
        for y_col in y_vars:

            data = df[[x_col, y_col]].replace([np.inf, -np.inf], np.nan).dropna().copy()

            if len(data) < 20:
                continue

            y_raw_std = float(data[y_col].std())

            for method in methods:

                if method == "equal_count_bin":
                    x_curve, y_curve = fit_binned_curve(
                        data,
                        x_col,
                        y_col,
                        n_bins=n_bins,
                        bin_type="equal_count"
                    )

                elif method == "equal_width_bin":
                    x_curve, y_curve = fit_binned_curve(
                        data,
                        x_col,
                        y_col,
                        n_bins=n_bins,
                        bin_type="equal_width"
                    )

                elif method == "lowess":
                    x_curve, y_curve = fit_lowess_curve(
                        data,
                        x_col,
                        y_col,
                        frac=lowess_frac,
                        n_grid=n_grid
                    )

                elif method == "polynomial_2":
                    x_curve, y_curve = fit_polynomial_curve(
                        data,
                        x_col,
                        y_col,
                        degree=2,
                        n_grid=n_grid
                    )

                elif method == "polynomial_3":
                    x_curve, y_curve = fit_polynomial_curve(
                        data,
                        x_col,
                        y_col,
                        degree=3,
                        n_grid=n_grid
                    )

                else:
                    raise ValueError(f"Unknown method: {method}")

                if len(x_curve) < 5:
                    result = {"shape_type": "Insufficient data"}
                else:
                    result = classify_curve_shape(
                        x_curve=x_curve,
                        y_curve=y_curve,
                        y_raw_std=y_raw_std
                    )

                key = (model_name, x_col, y_col, method)

                curve_dict[key] = {
                    "x_curve": x_curve,
                    "y_curve": y_curve,
                    "x_raw": data[x_col].values,
                    "y_raw": data[y_col].values,
                }

                rows.append({
                    "model": model_name,
                    "x": x_col,
                    "y": y_col,
                    "method": method,
                    "n_raw": len(data),
                    **result
                })

    result_df = pd.DataFrame(rows)

    return result_df, curve_dict


# =========================================================
# 4. Run form classification for two models
# =========================================================

def run_form_classification_two_models(
    classic_df,
    lpj_df,
    x_vars=("precipitation", "lai"),
    y_vars=("evapotrans", "tran", "evspsblveg", "evspsblsoi"),
):
    """
    Run form classification for CLASSIC and LPJ-GUESS.
    """

    classic_results, classic_curves = run_form_classification_for_dataset(
        df=classic_df,
        model_name="CLASSIC",
        x_vars=x_vars,
        y_vars=y_vars
    )

    lpj_results, lpj_curves = run_form_classification_for_dataset(
        df=lpj_df,
        model_name="LPJ-GUESS",
        x_vars=x_vars,
        y_vars=y_vars
    )

    result_df = pd.concat(
        [classic_results, lpj_results],
        ignore_index=True
    )

    curve_dict = {}
    curve_dict.update(classic_curves)
    curve_dict.update(lpj_curves)

    return result_df, curve_dict


# =========================================================
# 5. Summarise method agreement
# =========================================================

def summarise_form_agreement(result_df):
    """
    Summarise whether different fitting methods give consistent form labels.
    """

    summary_rows = []

    group_cols = ["model", "x", "y"]

    for keys, group in result_df.groupby(group_cols):
        model, x_col, y_col = keys

        valid_group = group[group["shape_type"] != "Insufficient data"].copy()

        if len(valid_group) == 0:
            final_shape = "Insufficient data"
            agreement_ratio = np.nan
            n_methods = 0
            n_agree = 0

        else:
            counts = valid_group["shape_type"].value_counts()
            final_shape = counts.index[0]
            n_agree = int(counts.iloc[0])
            n_methods = int(len(valid_group))
            agreement_ratio = float(n_agree / n_methods)

        summary_rows.append({
            "model": model,
            "x": x_col,
            "y": y_col,
            "final_shape_by_majority": final_shape,
            "method_agreement_ratio": agreement_ratio,
            "n_methods": n_methods,
            "n_agree": n_agree,
            "method_shapes": dict(valid_group["shape_type"].value_counts())
        })

    return pd.DataFrame(summary_rows)


# =========================================================
# 6. Plot fitted curves with form labels
# =========================================================

def plot_form_curves(
    result_df,
    curve_dict,
    model_name,
    x_col,
    y_col,
    methods=("equal_count_bin", "equal_width_bin", "lowess", "polynomial_2", "polynomial_3"),
    sample_size=8000,
    point_size=2,
    point_alpha=0.10,
    figsize=(16, 9)
):
    """
    Plot raw scatter and fitted curves for one model-x-y relationship.
    """

    n_methods = len(methods)
    n_cols = min(3, n_methods)
    n_rows = int(np.ceil(n_methods / n_cols))

    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=figsize,
        squeeze=False
    )

    for i, method in enumerate(methods):
        ax = axes[i // n_cols, i % n_cols]

        key = (model_name, x_col, y_col, method)

        if key not in curve_dict:
            ax.set_title(f"{method}\nNo curve")
            ax.axis("off")
            continue

        curve_info = curve_dict[key]

        x_raw = curve_info["x_raw"]
        y_raw = curve_info["y_raw"]
        x_curve = curve_info["x_curve"]
        y_curve = curve_info["y_curve"]

        if len(x_raw) > sample_size:
            rng = np.random.default_rng(42)
            idx = rng.choice(len(x_raw), size=sample_size, replace=False)
            x_plot = x_raw[idx]
            y_plot = y_raw[idx]
        else:
            x_plot = x_raw
            y_plot = y_raw

        ax.scatter(
            x_plot,
            y_plot,
            s=point_size,
            alpha=point_alpha
        )

        ax.plot(
            x_curve,
            y_curve,
            linewidth=2.2
        )

        shape = result_df[
            (result_df["model"] == model_name)
            & (result_df["x"] == x_col)
            & (result_df["y"] == y_col)
            & (result_df["method"] == method)
        ]["shape_type"]

        shape_label = shape.iloc[0] if len(shape) > 0 else "Unknown"

        ax.set_title(f"{method}\n{shape_label}", fontsize=11)
        ax.set_xlabel(x_col)
        ax.set_ylabel(y_col)

    # Remove empty axes
    for j in range(n_methods, n_rows * n_cols):
        axes[j // n_cols, j % n_cols].axis("off")

    fig.suptitle(
        f"{model_name}: {x_col} → {y_col}",
        fontsize=15,
        y=1.02
    )

    plt.tight_layout()
    plt.show()


# =========================================================
# 7. Example usage
# =========================================================

# Example:
# classic_df = pd.read_csv("classic_with_climate_zones_filtered_30yr_mean.csv")
# lpj_df = pd.read_csv("lpj_guess_with_climate_zones_filtered_30yr_mean.csv")

# Select variables according to your current table column names
# x_vars = ("precipitation", "lai")
# y_vars = ("evapotrans", "tran", "evspsblveg", "evspsblsoi")

# form_results_df, form_curve_dict = run_form_classification_two_models(
#     classic_df=classic_df,
#     lpj_df=lpj_df,
#     x_vars=x_vars,
#     y_vars=y_vars
# )

# form_summary_df = summarise_form_agreement(form_results_df)

# Display detailed results
# form_results_df[
#     [
#         "model", "x", "y", "method", "shape_type",
#         "net_change", "pos_fraction", "neg_fraction", "flat_fraction",
#         "early_increase_strength", "late_increase_strength",
#         "early_decrease_strength", "late_decrease_strength",
#         "saturating_increase", "saturating_decrease",
#         "n_sign_changes", "roughness_ratio"
#     ]
# ]

# Display majority-vote summary
# form_summary_df

# Plot one example
# plot_form_curves(
#     result_df=form_results_df,
#     curve_dict=form_curve_dict,
#     model_name="CLASSIC",
#     x_col="precipitation",
#     y_col="tran"
# )

In [3]:

# =========================================================
# 7. Example usage
# =========================================================

# Example:
classic_df = pd.read_csv("classic_with_climate_zones_filtered_30yr_mean.csv")
lpj_df = pd.read_csv("lpj_guess_with_climate_zones_filtered_30yr_mean.csv")

# Select variables according to your current table column names
x_vars = ("precipitation", "lai")
y_vars = ("evapotrans", "tran", "evspsblveg", "evspsblsoi")

form_results_df, form_curve_dict = run_form_classification_two_models(
    classic_df=classic_df,
    lpj_df=lpj_df,
    x_vars=x_vars,
    y_vars=y_vars
)

form_summary_df = summarise_form_agreement(form_results_df)

# Display detailed results
form_results_df[
    [
        "model", "x", "y", "method", "shape_type",
        "net_change", "pos_fraction", "neg_fraction", "flat_fraction",
        "early_increase_strength", "late_increase_strength",
        "early_decrease_strength", "late_decrease_strength",
        "saturating_increase", "saturating_decrease",
        "n_sign_changes", "roughness_ratio"
    ]
]

# Display majority-vote summary
# form_summary_df

# Plot one example
plot_form_curves(
    result_df=form_results_df,
    curve_dict=form_curve_dict,
    model_name="CLASSIC",
    x_col="precipitation",
    y_col="tran"
)

FileNotFoundError: [Errno 2] No such file or directory: 'classic_with_climate_zones_filtered_30yr_mean.csv'